# DeepRL-RecSys-Platform: End-to-End Demo 🚀

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aalopez76/DeepRL-RecSys-Platform/blob/main/examples/demo_train_eval_serve.ipynb)

Bienvenido a la demostración paso a paso de **DeepRL-RecSys-Platform**. Esta plataforma permite entrenar agentes de Reinforcement Learning para sistemas de recomendación, evaluarlos de manera segura usando **Off-Policy Evaluation (OPE)** y servirlos en tiempo real.

**Repositorio Oficial:** [github.com/aalopez76/DeepRL-RecSys-Platform](https://github.com/aalopez76/DeepRL-RecSys-Platform)

En este notebook aprenderemos a:
1. Preparar el entorno e instalar dependencias.
2. Generar datos sintéticos para la prueba.
3. Configurar y entrenar un agente Soft Actor-Critic (SAC).
4. Evaluar el agente con estimadores robustos (IPS, DR) y semáforo de fiabilidad.
5. Exportar el artefacto resultante.
6. Servir el modelo con una API REST y consumir predicciones.


## 1. Instalación y Configuración ⚙️

Si ejecutas esto en Colab, automáticamente clonaremos el repositorio e instalaremos la plataforma. Si estás en local, asegúrate de haber seguido las instrucciones del `README.md` (usando Poetry o `pip install -e .[ui,torch]`).

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ejecutando en Google Colab. Clonando repositorio...")
    !git clone https://github.com/aalopez76/DeepRL-RecSys-Platform.git
    %cd DeepRL-RecSys-Platform
    print("\nInstalando dependencias necesarias...")
    # Instalar pandas primero (ya está preinstalado en Colab, pero por si acaso)
    !pip install pandas --quiet
    # Instalar la plataforma sin modo editable para evitar la compilación fallida en PyPI
    !pip install .[ui,torch] --quiet
else:
    print("Ejecutando localmente. Asegúrate de tener el entorno virtual activo.")

try:
    import deeprl_recsys
    print(f"\n✅ DeepRL-RecSys-Platform instalada. Versión: {deeprl_recsys.__version__}")
except ImportError as e:
    print("\n❌ Error al importar deeprl_recsys. La instalación falló.")
    raise e


## 2. Generación de Datos Sintéticos 📊

Vamos a generar un dataset sintético estructurado bajo el esquema `bandit_v1`. Tendremos listas de contextos, acciones tomadas (items recomendados), recompensas observadas (clics/compras) y la propensión de la política original (qué tan probable era tomar esa acción).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Aseguramos que la carpeta de datos exista
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

# Parámetros de la simulación
num_rows = 10000
num_items = 50

np.random.seed(42)

df_synthetic = pd.DataFrame({
    "context": [f'{{"user_id": {np.random.randint(1, 100)}, "age": {np.random.randint(18, 60)}}}' for _ in range(num_rows)],
    "action": np.random.randint(0, num_items, size=num_rows),
    "reward": np.random.binomial(1, p=0.15, size=num_rows),  # Recompensa binaria (ej. 15% CTR global promedio)
    "propensity": np.random.uniform(0.1, 0.9, size=num_rows), # Simulación de una logging policy estocástica
    "timestamp": pd.date_range(start="2026-01-01", periods=num_rows, freq="min")
})

dataset_path = data_dir / "synthetic_demo.parquet"
df_synthetic.to_parquet(dataset_path)

print(f"Dataset sintético generado en {dataset_path} con {len(df_synthetic)} filas.")
df_synthetic.head()

## 3. Configuración del Experimento ⚗️

La configuración principal de un experimento RL en este framework se realiza mediante archivos YAML. Vamos a crear dinámicamente el archivo `sac_demo.yaml`, indicando que queremos usar nuestro nuevo dataset y probar un agente **Soft Actor-Critic (SAC)**.

In [ ]:
import yaml

configs_dir = Path("./configs")
configs_dir.mkdir(parents=True, exist_ok=True)

config = {
    "seed": 42,
    "agent": {
        "name": "sac",
        "hyperparams": {
            "learning_rate": 0.001,
            "gamma": 0.99,
            "alpha": 0.5, # Ajuste fino para la temperatura de distribución
            "tau": 0.005,
            "batch_size": 128,
            "buffer_size": 50000,
        }
    },
    "dataset": {
        # Utilizamos ruta relativa asegurando portabilidad
        "path": "./data/synthetic_demo.parquet",
        "format": "parquet",
        "schema_version": "bandit_v1",
        "propensity_policy": "mark_unreliable"
    },
    "training": {
        "max_steps": 500,
        "eval_interval": 100,
        "checkpoint_interval": 250
    }
}

config_path = configs_dir / "sac_demo.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Archivo YAML de configuración guardado en {config_path}\n")
!cat {config_path}

## 4. Entrenamiento del Agente 🧠

El entrenamiento se orquesta automáticamente a través de la CLI del framework. Ejecutaremos el pipeline `train` pasándole nuestro configuración.

In [ ]:
# Ejecutar entrenamiento. (El comando '!' llama directamente a la shell del sistema)
!python -m deeprl_recsys.cli train --config ./configs/sac_demo.yaml

## 5. Evaluación Off-Policy (OPE) 👀

En vez de llevar un modelo inmediatamente a A/B Test (lo cual es costoso y riesgoso), la evaluación *Off-Policy* aprovecha los logs históricos recolectados para estimar cuál hubiese sido el ROI del modelo propuesto.

Lanzamos el pipeline de evaluación:

In [ ]:
!python -m deeprl_recsys.cli evaluate --config ./configs/sac_demo.yaml

### 5.1 Semáforo de Fiabilidad y Resultados OPE 🚦

Vamos a inspeccionar los resultados. Cargas el archivo JSON de evaluación y pintamos el "Semáforo de Fiabilidad" de la plataforma, seguido de un gráfico comparativo de los diferentes estimadores de recompensa.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

# Buscar el reporte OPE más reciente
report_files = glob.glob("./artifacts/models/**/ope_report.json", recursive=True)
if not report_files:
    report_files = glob.glob("./artifacts/checkpoints/**/ope_report.json", recursive=True)

if report_files:
    latest_report = max(report_files, key=os.path.getmtime)
    print(f"📄 Cargando reporte OPE desde: {latest_report}")
    try:
        with open(latest_report, "r") as f:
            ope_data = json.load(f)
    except json.JSONDecodeError as e:
        print(f"❌ Error al decodificar JSON en {latest_report}: {e}")
        ope_data = {}
else:
    raise FileNotFoundError("No se encontró ningún archivo ope_report.json en artifacts/")

verdict = ope_data.get("verdict", {}).get("severity", "unknown").upper()
estimates = ope_data.get("estimates", {})

# Verificación robusta de importance_weights
if "importance_weights" in ope_data and ope_data["importance_weights"]:
    weights = ope_data["importance_weights"]
    w_arr = np.array(weights)
    ess = (np.sum(w_arr) ** 2) / (np.sum(w_arr ** 2) + 1e-9)
    ess_ratio = ess / max(len(w_arr), 1)
else:
    weights = []
    w_arr = np.array([])
    ess = 0.0
    ess_ratio = 0.0
    print("⚠️ No se encontraron pesos de importancia (importance_weights); el ESS será 0.0.")

# 1. Semáforo de Fiabilidad
colors = {"PASS": "#28a745", "WARN": "#ffc107", "FAIL": "#dc3545", "UNKNOWN": "#6c757d"}
text_labels = {"PASS": "Reliable", "WARN": "Warning", "FAIL": "Unreliable", "UNKNOWN": "Unknown"}
color = colors.get(verdict, colors["UNKNOWN"])
label = text_labels.get(verdict, text_labels["UNKNOWN"])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Semáforo
ax[0].add_patch(plt.Circle((0.5, 0.5), 0.3, color=color, alpha=0.9))
ax[0].text(0.5, 0.5, label, color="white", fontsize=24, fontweight="bold", 
           horizontalalignment='center', verticalalignment='center')
ax[0].set_title(f"Diagnosis Severity: {verdict}\nESS: {ess:.1f} ({ess_ratio:.1%} retention)", fontsize=14)
ax[0].axis("off")

# 2. Gráfico de Barras de Estimadores OPE
est_names = list(estimates.keys())
est_vals = list(estimates.values())

if est_names:
    ax[1].bar(est_names, est_vals, color="#007bff", alpha=0.8)
    for i, v in enumerate(est_vals):
        ax[1].text(i, v + (max(est_vals)*0.02), f"{v:.4f}", ha='center', fontweight='bold')
    ax[1].axhline(y=0.15, color="red", linestyle="--", label="Baseline / Logging Policy CTR (~0.15)")

ax[1].set_title("Off-Policy Estimates Comparability", fontsize=14)
ax[1].set_ylabel("Estimated Cumulative Reward")
ax[1].legend()

plt.tight_layout()
plt.show()

print("Interpretación:")
print("- Si el ESS retenido es muy bajo y arroja FAIL/WARN, significa que las predicciones del modelo se distancian excesivamente del dataset original.")
print("- Un veredicto verde (PASS) asegura que la estimación de IPS y Doubly Robust es matemáticamente sólida acorde a los bounds estadísticos.")


## 6. Exportación del Artefacto 📦

Transformamos los resultados del archivo base a un artefacto persistido en el registro interno.

In [ ]:
!python -m deeprl_recsys.cli export --config ./configs/sac_demo.yaml
print("\n--- Visualizando el interior del Artefacto Exportado ---")
!ls -lh ./artifacts/models/latest

## 7. Servir el Modelo (Inferencia API REST) 🚀

Levantaremos el servidor uvicorn en segundo plano, verificaremos que esté saludable vía `/health`, ¡y realizaremos nuestra primera inferencia de recomendación pasando un contexto de juguete!

Finalmente, terminaremos el proceso del servidor para no ahogar el puerto en el plano virtual.

In [ ]:
import subprocess
import time
import requests

# Iniciar el servidor 
server_process = subprocess.Popen(
    [sys.executable, "-m", "deeprl_recsys.cli", "serve", "--artifact-dir", "./artifacts/models/latest", "--port", "8000"], 
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print("Esperando que el servidor inicie...")

try:
    # Health check Retry Loop
    server_ready = False
    for attempt in range(15):
        try:
            res = requests.get("http://127.0.0.1:8000/health")
            if res.status_code == 200:
                server_ready = True
                print("\n✅ Servidor listo y respondiendo correctamente.")
                break
        except requests.ConnectionError:
            pass
        print(".", end="")
        time.sleep(1)
        
    if not server_ready:
        raise RuntimeError("El servidor no respondió a tiempo. Revisa los logs de la consola.")
        
    # Prueba de Recomendación
    print("\n--- Ejecutando Endpoint de Recomendación ---")
    req_payload = {
        "context": {"user_id": 777, "time_active": 300},
        "candidates": [10, 25, 33, 41, 45, 12, 50, 2],  # Qué item presentar al usuario?
        "k": 5
    }
    
    response = requests.post("http://127.0.0.1:8000/recommend", json=req_payload)
    
    if response.status_code == 200:
        predictions = response.json()["recommendations"]
        print("\n🔮 [Predicciones RL] Top 5 Recomendaciones:")
        for i, item in enumerate(predictions):
             print(f"  #{i+1} -> Item ID: {item['item_id']} | Score (Est. Probability): {item['score']:.4f}")
    else:
        print(f"❌ Fallo de servidor: {response.text}")

finally:
    # ASEGURARNOS de terminar el servidor siempre, para no dejar procesos zombis en colab / local
    print("\nApagando servidor de inferencia...")
    server_process.terminate()
    server_process.wait(timeout=5)
    print("Servidor finalizado exitosamente.")

## 8. ¿Qué sigue? Dashboard Interactivo

El proyecto también provee una interfaz Streamlit profesional de análisis de OPE que permite revisar los pesos y reportes a fondo, y tener un **Playground Textual** de evaluación directa del motor servido.

En una máquina local, sencillamente lanza:
```bash
deeprl-recsys ui
```

Esto abrirá un portal en tu navegador para auditar estos componentes visualmente.
---
*Plataforma Desarrollada para validación y deployment E2E seguro de Reinforcement Learning sobre OPE.*

## 9. Limpieza de Entorno (Opcional) 🧹

Ejecuta esta celda si deseas eliminar los archivos de datos sintéticos, configuraciones locales y artefactos generados durante este notebook para liberar espacio.

In [ ]:
# Celda de limpieza opcional (ejecutar solo si se desea borrar los archivos generados)
import shutil

respuesta = input("¿Borrar los datos, configuraciones y artefactos generados? (s/n): ")
if respuesta.lower() == 's':
    shutil.rmtree("./data", ignore_errors=True)
    shutil.rmtree("./configs", ignore_errors=True)
    shutil.rmtree("./artifacts", ignore_errors=True)
    print("✅ Archivos eliminados.")
else:
    print("Limpieza omitida.")
